# 🌱 HydroGrow AI — Phase 1: Data Understanding & Cleaning

---

**Full Title:** HydroGrow AI: An Intelligent Decision Support System for Hydroponic Lettuce Growth Prediction, Disease Prevention and Growth Optimization

**Notebook:** `01_Data_Understanding_and_Cleaning.ipynb`  
**Author:** HydroGrow AI Team  
**Date:** 2026-07-15  
**Version:** 1.0

---

## 1 · Project Introduction

### 1.1 Objective

HydroGrow AI is designed to become a production-grade **AI-powered Hydroponic Assistant** capable of:

| Capability | Status |
|---|---|
| Understanding hydroponic lettuce data | 🔄 Phase 1 (this notebook) |
| Predicting plant growth | 📋 Planned |
| Predicting plant requirements | 📋 Planned |
| Supporting disease prevention | 📋 Planned |
| Providing intelligent recommendations | 📋 Planned |
| Machine Learning integration | 📋 Planned |
| ChatGPT-like web application | 📋 Planned |

**This notebook focuses exclusively on Phase 1**: data understanding, quality assessment, and safe cleaning.

### 1.2 Dataset Overview

Three independent hydroponic lettuce experiments are available, each stored as an Excel workbook containing multiple sheets:

| Dataset | Description |
|---|---|
| **Experiment 1** | Environmental conditions & plant measurements (6 sheets) |
| **Experiment 2** | Environmental conditions & plant measurements (7 sheets) |
| **Experiment 3** | Environmental conditions & plant measurements (5 sheets) |

Each workbook typically contains sheets for:
- **Seedlings measurements** — baseline plant biometrics at transplant
- **Water quality (Sensor)** — continuous pH, EC, TDS, temperature, humidity, CO₂
- **Water quality (Portable)** — manual spot-check measurements across replicates
- **Nutrients & Water consumptions** — nutrient dosing and water usage logs
- **Harvest measurements** — final harvest biometrics per replicate/treatment
- Additional sheets (Head diameter, Taste tests, Survey forms) in some experiments

### 1.3 Expected Workflow

```
Phase 1 (This Notebook)
├── Load all raw datasets
├── Understand structure & semantics
├── Assess data quality comprehensively
├── Apply only safe, reversible cleaning
├── Validate cleaning outcomes
├── Generate data dictionary
└── Save cleaned datasets → data/processed/
```

> ⚠️ **Guiding Principle:** We never delete rows due to outliers, never invent missing values, and never remove suspicious values. If something looks questionable, we **flag it** and document it for expert review.

---

## 2 · Import Libraries

In [9]:
"""
Import only the libraries required for Phase 1.
No ML / DL / visualization-heavy imports at this stage.
"""

import os
import sys
import logging
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.float_format", "{:.4f}".format)

# ---------------------------------------------------------------------------
# Logging — gives us a persistent audit trail inside the notebook
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("HydroGrow")
logger.info("Libraries imported successfully.")

# ---------------------------------------------------------------------------
# Project paths (relative — never hardcoded absolutes)
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path(os.getcwd()).parent  # notebooks/ → project root
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Project root : {PROJECT_ROOT}")
logger.info(f"Raw data dir : {RAW_DATA_DIR}")
logger.info(f"Processed dir: {PROCESSED_DATA_DIR}")

print(f"\n✅ Python {sys.version}")
print(f"✅ pandas {pd.__version__}  |  numpy {np.__version__}")

2026-07-15 19:47:37,441 | INFO | Libraries imported successfully.
2026-07-15 19:47:37,443 | INFO | Project root : e:\HydroGrow-AI
2026-07-15 19:47:37,445 | INFO | Raw data dir : e:\HydroGrow-AI\data\raw
2026-07-15 19:47:37,445 | INFO | Processed dir: e:\HydroGrow-AI\data\processed



✅ Python 3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]
✅ pandas 3.0.3  |  numpy 2.4.4


---

## 3 · Load Data

Each experiment workbook contains **multiple sheets** with different structures (multi-row headers, merged cells, sub-tables). We load every sheet of every workbook into a structured dictionary so no information is lost.

In [10]:
import sys

print(sys.executable)
print(sys.version)

c:\Python314\python.exe
3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]


In [11]:
from pathlib import Path
from typing import Dict, List
import pandas as pd

# -------------------------------
# Project Paths
# -------------------------------
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(f"Raw data folder not found: {RAW_DATA_DIR}")

# -------------------------------
# Discover Excel Files
# -------------------------------
def discover_excel_files(directory: Path) -> List[Path]:
    """Discover all .xlsx files in the given directory."""
    files = sorted(directory.glob("*.xlsx"))

    if not files:
        raise FileNotFoundError(f"No .xlsx files found in {directory}")

    print(f"\n✅ Found {len(files)} Excel file(s):")

    for f in files:
        print(f"   - {f.name}")

    return files


# -------------------------------
# Load All Sheets
# -------------------------------
def load_all_sheets(filepath: Path) -> Dict[str, pd.DataFrame]:
    """
    Load every sheet from an Excel workbook.
    """

    xl = pd.ExcelFile(filepath, engine="openpyxl")

    sheets = {}

    for name in xl.sheet_names:

        sheets[name] = pd.read_excel(
            filepath,
            sheet_name=name,
            header=None,
            engine="openpyxl"
        )

        print(
            f"      ✓ {name:<40} "
            f"{sheets[name].shape[0]} rows × {sheets[name].shape[1]} cols"
        )

    return sheets


# -------------------------------
# Discover and Load
# -------------------------------
excel_files = discover_excel_files(RAW_DATA_DIR)

experiments_raw = {}

for idx, fpath in enumerate(excel_files, start=1):

    label = f"Experiment {idx}"

    print(f"\n📂 Loading {label}: {fpath.name}")

    experiments_raw[label] = load_all_sheets(fpath)

# -------------------------------
# Summary
# -------------------------------
print("\n" + "=" * 80)
print("LOADING SUMMARY")
print("=" * 80)

for exp_label, sheets in experiments_raw.items():

    total_rows = sum(df.shape[0] for df in sheets.values())

    print(f"\n📁 {exp_label}")
    print(f"   Total Sheets : {len(sheets)}")
    print(f"   Total Rows   : {total_rows:,}")

    for sheet_name, df in sheets.items():
        print(f"   ├── {sheet_name:<40} {df.shape[0]:>6} × {df.shape[1]:>3}")

print("\n✅ All datasets loaded successfully.")


✅ Found 3 Excel file(s):
   - EXP.1. Environmental Conditions and Plant Measurements.xlsx
   - EXP.2. Environmental Conditions and Plant Measurements.xlsx
   - EXP.3. Environmental Conditions and Plant Measurements.xlsx

📂 Loading Experiment 1: EXP.1. Environmental Conditions and Plant Measurements.xlsx
      ✓ Seedlings measurements                   12 rows × 11 cols
      ✓ Water quality measurments Senso          643 rows × 11 cols
      ✓ Water quality parametersPortabl          36 rows × 56 cols
      ✓ Nutrients  Water consumptions            26 rows × 23 cols
      ✓ Head diameter                            73 rows × 10 cols
      ✓ Harvest measurements 842024              117 rows × 11 cols

📂 Loading Experiment 2: EXP.2. Environmental Conditions and Plant Measurements.xlsx
      ✓ Seedlings measurements                   14 rows × 11 cols
      ✓ Water quality measurments Senso          720 rows × 11 cols
      ✓ Water quality parametersPortabl          32 rows × 56 cols
   

### 3.1 Parse Sheets into Clean DataFrames

The raw Excel sheets use **multi-row headers**, **merged cells**, and **sub-tables** side-by-side. Below we parse each sheet type into properly-headed DataFrames using dedicated parser functions.

> We apply **minimal parsing only** — setting correct headers and stripping structural artifacts. No data values are modified.

In [12]:
# ==========================================================================
# Sheet Parsers — one per sheet type
# ==========================================================================

def safe_str_row(series: pd.Series) -> list:
    """
    Convert a pandas Series (row) into a list of clean Python strings.
    Handles pandas 3.x nullable string dtype where .astype(str)
    preserves NaN as float('nan') instead of the string 'nan'.
    Returns '' for any missing/NaN/NaT values.
    """
    return [str(v).strip() if pd.notna(v) else "" for v in series]


def parse_seedlings(df_raw: pd.DataFrame, exp_label: str) -> pd.DataFrame:
    """
    Parse the Seedlings measurements sheet.
    Row 0 is a title row; Row 1 contains column headers; Data starts at Row 2.
    """
    headers = safe_str_row(df_raw.iloc[1])
    data = df_raw.iloc[2:].copy()
    data.columns = headers
    data = data.reset_index(drop=True)
    data.insert(0, "experiment", exp_label)
    return data


def parse_sensor_water_quality(df_raw: pd.DataFrame, exp_label: str) -> pd.DataFrame:
    """
    Parse the Water quality measurements (Sensor) sheet.
    Row 0: replicate label; Row 1: column headers.
    Columns 0-5: Water quality; Columns 6-10: Air parameters.
    Data starts at Row 2.
    """
    # Extract replicate label from row 0 (e.g., 'Replicate 1 T1')
    replicate_label = str(df_raw.iloc[0, 2]).strip() if pd.notna(df_raw.iloc[0, 2]) else "Unknown"

    headers = safe_str_row(df_raw.iloc[1])
    data = df_raw.iloc[2:].copy()
    data.columns = headers
    data = data.reset_index(drop=True)
    data.insert(0, "replicate", replicate_label)
    data.insert(0, "experiment", exp_label)

    # The sheet has two Date/Time column pairs (water + air)
    # Rename to disambiguate
    cols = list(data.columns)
    seen = {}
    new_cols = []
    for c in cols:
        if c in seen:
            seen[c] += 1
            new_cols.append(f"{c}_{seen[c]}")
        else:
            seen[c] = 0
            new_cols.append(c)
    data.columns = new_cols
    return data


def parse_portable_water_quality(df_raw: pd.DataFrame, exp_label: str) -> pd.DataFrame:
    """
    Parse the Water quality parameters (Portable) sheet.
    This has a complex 3-row header: Row 0 = time slots & air params,
    Row 1 = replicate labels, Row 2 = measurement names.
    Data starts at Row 3.
    """
    row0 = safe_str_row(df_raw.iloc[0])
    row1 = safe_str_row(df_raw.iloc[1])
    row2 = safe_str_row(df_raw.iloc[2])

    # Build composite column names: {time}_{replicate}_{measurement}
    composite_cols = []
    current_time = "unknown"
    current_replicate = "unknown"

    for i in range(len(row0)):
        r0 = row0[i]
        r1 = row1[i]
        r2 = row2[i]

        if r0 and r0 not in ("nan", "NaT") and "Air" not in r0 and "Date" not in r0:
            current_time = r0.replace(":", "")
        if r0 in ("Date", "Date "):
            current_time = "date"
        if r0 and "Air" in r0:
            current_replicate = "Air"

        if r1 and r1 not in ("nan", "NaT") and (not r0 or "Air" not in r0):
            current_replicate = r1.replace(" ", "_")

        if r2 and r2 not in ("nan", "NaT"):
            measurement = r2
        else:
            measurement = r0 if (r0 and r0 not in ("nan", "NaT")) else f"col_{i}"

        if current_time == "date":
            col_name = "date"
        elif current_replicate == "Air":
            col_name = f"{current_time}_air_{measurement}"
        else:
            col_name = f"{current_time}_{current_replicate}_{measurement}"

        composite_cols.append(col_name)

    # Ensure uniqueness
    seen = {}
    unique_cols = []
    for c in composite_cols:
        c_clean = c.replace(" ", "_").replace(".", "").lower()
        if c_clean in seen:
            seen[c_clean] += 1
            unique_cols.append(f"{c_clean}_{seen[c_clean]}")
        else:
            seen[c_clean] = 0
            unique_cols.append(c_clean)

    data = df_raw.iloc[3:].copy()
    data.columns = unique_cols
    data = data.reset_index(drop=True)
    data.insert(0, "experiment", exp_label)
    return data


def parse_nutrients_water(df_raw: pd.DataFrame, exp_label: str) -> Dict[str, pd.DataFrame]:
    """
    Parse the Nutrients & Water consumptions sheet.
    Contains 2 or 3 side-by-side sub-tables separated by blank columns.
    Returns a dict of parsed sub-tables.
    """
    row0 = df_raw.iloc[0]

    # Find sub-table boundaries from row 0 headers
    sub_tables = {}
    current_label = None
    start_col = None

    for i, val in enumerate(row0):
        val_str = str(val).strip() if pd.notna(val) else ""
        if val_str and val_str not in ("nan", "NaT"):
            # Close previous sub-table
            if current_label is not None:
                sub_tables[current_label] = (start_col, i - 1)
            current_label = val_str
            start_col = i
    # Close last sub-table
    if current_label is not None:
        sub_tables[current_label] = (start_col, len(row0) - 1)

    result = {}
    for label, (sc, ec) in sub_tables.items():
        cols_range = list(range(sc, ec + 1))
        sub_df = df_raw.iloc[1:, cols_range].copy()

        # Drop fully empty columns
        sub_df = sub_df.dropna(axis=1, how="all")

        if sub_df.empty or sub_df.shape[1] == 0:
            continue

        # Row 0 of sub_df has replicate headers
        header_vals = safe_str_row(sub_df.iloc[0])
        sub_data = sub_df.iloc[1:].copy()

        # Build column names
        new_cols = []
        for j, h in enumerate(header_vals):
            if j == 0 and ("date" in label.lower() or not h or h in ("nan", "NaT")):
                new_cols.append("date")
            elif not h or h in ("nan", "NaT"):
                new_cols.append(f"col_{j}")
            else:
                new_cols.append(h.replace(" ", "_"))

        sub_data.columns = new_cols
        sub_data = sub_data.reset_index(drop=True)
        sub_data.insert(0, "experiment", exp_label)

        # Clean label for dict key
        clean_label = label.replace(" ", "_").replace(".", "").lower()
        result[clean_label] = sub_data

    return result


def parse_harvest(df_raw: pd.DataFrame, exp_label: str) -> pd.DataFrame:
    """
    Parse the Harvest measurements sheet.
    These sheets contain multiple replicate/treatment sub-tables stacked vertically.
    Row pattern: [Replicate-Treatment label], [Column headers], [Data rows], [blank], repeat.
    """
    all_records = []
    current_system = None
    header_row = None

    # Determine if this is EXP3 format (has 'System' column)
    has_system_col = False
    for i in range(min(5, len(df_raw))):
        row_vals = [str(v).strip() if pd.notna(v) else "" for v in df_raw.iloc[i]]
        if "System" in row_vals or "System " in row_vals:
            has_system_col = True
            break

    if has_system_col:
        # EXP3 format: single table with a 'System' column
        for i in range(len(df_raw)):
            row_vals = [str(v).strip() if pd.notna(v) else "" for v in df_raw.iloc[i]]
            if "Plant No." in row_vals or "Plant No" in row_vals:
                header_row = i
                break

        if header_row is not None:
            headers = safe_str_row(df_raw.iloc[header_row])
            data = df_raw.iloc[header_row + 1:].copy()
            data.columns = headers
            data = data.dropna(how="all").reset_index(drop=True)

            # Forward-fill the System column
            if "System " in data.columns:
                data = data.rename(columns={"System ": "system"})
            elif "System" in data.columns:
                data = data.rename(columns={"System": "system"})
            if "system" in data.columns:
                data["system"] = data["system"].replace("", np.nan).ffill()
            data.insert(0, "experiment", exp_label)
            return data

    # EXP1/EXP2 format: multiple sub-tables separated by replicate headers
    i = 0
    while i < len(df_raw):
        row_vals = [str(v).strip() if pd.notna(v) else "" for v in df_raw.iloc[i]]
        first_val = row_vals[0]

        # Detect replicate/treatment label rows (e.g., 'R1-T1', 'R2-T2')
        if first_val and first_val.startswith("R") and "T" in first_val and "Plant" not in first_val:
            current_system = first_val
            i += 1
            continue

        # Detect header row
        if "Plant No" in first_val or first_val == "Plant No.":
            header_row = [str(v).strip() if pd.notna(v) else "" for v in df_raw.iloc[i]]
            i += 1
            continue

        # Skip completely empty rows
        if all(v == "" or v == "nan" for v in row_vals):
            i += 1
            continue

        # Data row
        if header_row is not None and current_system is not None:
            record = dict(zip(header_row, df_raw.iloc[i].tolist()))
            record["system"] = current_system
            all_records.append(record)

        i += 1

    if all_records:
        result = pd.DataFrame(all_records)
        result.insert(0, "experiment", exp_label)
        return result

    # Fallback: return raw with experiment label
    df_raw.insert(0, "experiment", exp_label)
    return df_raw


def parse_head_diameter(df_raw: pd.DataFrame, exp_label: str) -> pd.DataFrame:
    """
    Parse the Head diameter sheet (EXP1 only).
    Row 0: day labels; Row 1: replicate label; Row 2: column headers.
    Data starts at Row 3. Structure: pairs of (Plant no, Head diameter) per day.
    """
    # Extract day labels from row 0
    day_labels = []
    for val in df_raw.iloc[0]:
        s = str(val).strip() if pd.notna(val) else ""
        if s and s not in ("nan", "NaT"):
            day_labels.append(s)

    # Identify replicate boundaries from row 1
    replicate_rows = []
    i = 1
    while i < len(df_raw):
        first_val = str(df_raw.iloc[i, 0]).strip() if pd.notna(df_raw.iloc[i, 0]) else ""
        if first_val.startswith("Replicate") or first_val.startswith("replicate"):
            replicate_rows.append((first_val, i))
        i += 1

    all_records = []

    for rep_idx, (rep_label, rep_row) in enumerate(replicate_rows):
        data_start = rep_row + 2
        if rep_idx + 1 < len(replicate_rows):
            data_end = replicate_rows[rep_idx + 1][1]
        else:
            data_end = len(df_raw)

        for row_i in range(data_start, data_end):
            row = df_raw.iloc[row_i]
            if pd.isna(row.iloc[0]):
                continue
            for day_idx, day_label in enumerate(day_labels):
                col_start = day_idx * 2
                if col_start + 1 < len(row):
                    record = {
                        "experiment": exp_label,
                        "replicate": rep_label,
                        "day": day_label,
                        "plant_no": row.iloc[col_start],
                        "head_diameter": row.iloc[col_start + 1],
                    }
                    all_records.append(record)

    return pd.DataFrame(all_records) if all_records else pd.DataFrame()


def parse_taste_test(df_raw: pd.DataFrame, exp_label: str) -> pd.DataFrame:
    """Parse Taste Test sheet — preserve as-is with experiment label."""
    df = df_raw.copy()
    df.insert(0, "experiment", exp_label)
    return df


def parse_form_responses(df_raw: pd.DataFrame, exp_label: str) -> pd.DataFrame:
    """Parse Form Responses sheet — preserve as-is with experiment label."""
    df = df_raw.copy()
    df.insert(0, "experiment", exp_label)
    return df

# ==========================================================
# Parser Registration Complete
# ==========================================================

import logging

# Create logger only if it doesn't already exist
try:
    logger
except NameError:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s"
    )
    logger = logging.getLogger("HydroGrowAI")

logger.info("✅ Sheet parser functions loaded successfully.")

print("=" * 70)
print("✅ All sheet parser functions have been defined successfully.")
print("=" * 70)


2026-07-15 19:47:52,321 | INFO | ✅ Sheet parser functions loaded successfully.


✅ All sheet parser functions have been defined successfully.


In [13]:
# ==========================================================================
# Route each sheet to its parser based on sheet name patterns
# ==========================================================================

def classify_and_parse_sheet(
    sheet_name: str, df_raw: pd.DataFrame, exp_label: str
) -> Tuple[str, Any]:
    """
    Classify a sheet by its name and dispatch to the appropriate parser.
    Returns (category, parsed_result) where parsed_result is a DataFrame
    or dict of DataFrames.
    """
    sn = sheet_name.lower().strip()

    if "seedling" in sn:
        return "seedlings", parse_seedlings(df_raw, exp_label)
    elif "senso" in sn or "sensor" in sn:
        return "sensor_water_quality", parse_sensor_water_quality(df_raw, exp_label)
    elif "portabl" in sn:
        return "portable_water_quality", parse_portable_water_quality(df_raw, exp_label)
    elif "nutrient" in sn:
        return "nutrients", parse_nutrients_water(df_raw, exp_label)
    elif "harvest" in sn:
        return "harvest", parse_harvest(df_raw, exp_label)
    elif "head" in sn and "diameter" in sn:
        return "head_diameter", parse_head_diameter(df_raw, exp_label)
    elif "taste" in sn or "test taste" in sn:
        return "taste_test", parse_taste_test(df_raw, exp_label)
    elif "form" in sn or "response" in sn:
        return "form_responses", parse_form_responses(df_raw, exp_label)
    else:
        logger.warning(f"Unknown sheet type: '{sheet_name}' — preserving raw.")
        df_raw.insert(0, "experiment", exp_label)
        return "unknown", df_raw


# ==========================================================================
# Parse all experiments
# ==========================================================================
parsed_data: Dict[str, Dict[str, Any]] = {}

for exp_label, sheets in experiments_raw.items():
    logger.info(f"Parsing {exp_label}...")
    parsed_data[exp_label] = {}
    for sheet_name, df_raw in sheets.items():
        category, result = classify_and_parse_sheet(sheet_name, df_raw.copy(), exp_label)

        # Nutrients parser returns a dict of sub-tables
        if isinstance(result, dict):
            for sub_key, sub_df in result.items():
                full_key = f"{category}_{sub_key}"
                parsed_data[exp_label][full_key] = sub_df
                logger.info(f"  ✓ {full_key}: {sub_df.shape}")
        else:
            parsed_data[exp_label][category] = result
            if isinstance(result, pd.DataFrame):
                logger.info(f"  ✓ {category}: {result.shape}")

print("\n✅ All sheets parsed successfully.")

2026-07-15 19:48:02,755 | INFO | Parsing Experiment 1...
2026-07-15 19:48:02,757 | INFO |   ✓ seedlings: (10, 12)
2026-07-15 19:48:02,761 | INFO |   ✓ sensor_water_quality: (641, 13)
2026-07-15 19:48:02,763 | INFO |   ✓ portable_water_quality: (33, 57)
2026-07-15 19:48:02,769 | INFO |   ✓ nutrients_date: (24, 2)
2026-07-15 19:48:02,770 | INFO |   ✓ nutrients_nutrient_solution_addition_(a+b)_ml: (24, 7)
2026-07-15 19:48:02,771 | INFO |   ✓ nutrients_acid_consumption_(ml): (24, 7)
2026-07-15 19:48:02,771 | INFO |   ✓ nutrients_water_consumption_l: (24, 7)
2026-07-15 19:48:02,783 | INFO |   ✓ head_diameter: (300, 5)
2026-07-15 19:48:02,794 | INFO |   ✓ harvest: (73, 13)
2026-07-15 19:48:02,795 | INFO | Parsing Experiment 2...
2026-07-15 19:48:02,796 | INFO |   ✓ seedlings: (12, 12)
2026-07-15 19:48:02,799 | INFO |   ✓ sensor_water_quality: (718, 13)
2026-07-15 19:48:02,802 | INFO |   ✓ portable_water_quality: (29, 57)
2026-07-15 19:48:02,805 | INFO |   ✓ nutrients_date: (24, 2)
2026-07-15


✅ All sheets parsed successfully.


In [14]:
# ==========================================================================
# Display parsed data summary per experiment
# ==========================================================================

def display_shape_summary(label: str, df: pd.DataFrame) -> None:
    """Display a formatted shape summary for a DataFrame."""
    print(f"  {label:<35s}  →  {df.shape[0]:>6,} rows × {df.shape[1]:>3} cols")


print("=" * 70)
print("PARSED DATA SUMMARY PER EXPERIMENT")
print("=" * 70)

for exp_label, tables in parsed_data.items():
    print(f"\n📂 {exp_label}")
    for tname, tdata in tables.items():
        if isinstance(tdata, pd.DataFrame):
            display_shape_summary(tname, tdata)
        else:
            print(f"  {tname:<35s}  →  (non-DataFrame: {type(tdata).__name__})")

PARSED DATA SUMMARY PER EXPERIMENT

📂 Experiment 1
  seedlings                            →      10 rows ×  12 cols
  sensor_water_quality                 →     641 rows ×  13 cols
  portable_water_quality               →      33 rows ×  57 cols
  nutrients_date                       →      24 rows ×   2 cols
  nutrients_nutrient_solution_addition_(a+b)_ml  →      24 rows ×   7 cols
  nutrients_acid_consumption_(ml)      →      24 rows ×   7 cols
  nutrients_water_consumption_l        →      24 rows ×   7 cols
  head_diameter                        →     300 rows ×   5 cols
  harvest                              →      73 rows ×  13 cols

📂 Experiment 2
  seedlings                            →      12 rows ×  12 cols
  sensor_water_quality                 →     718 rows ×  13 cols
  portable_water_quality               →      29 rows ×  57 cols
  nutrients_date                       →      24 rows ×   2 cols
  nutrients_nutrient_solution_addition_(a+b)_ml  →      24 rows ×   7 cols
  n

---

## 4 · Dataset Overview

For each experiment we display shape, head/tail rows, column names, data types, memory usage, and summary statistics.

In [15]:
def dataset_overview(df: pd.DataFrame, title: str) -> None:
    """
    Display a comprehensive overview of a DataFrame:
    shape, head, tail, columns, dtypes, memory, summary stats.
    """
    display(Markdown(f"### {title}"))

    # Shape
    display(Markdown(f"**Shape:** `{df.shape[0]:,}` rows × `{df.shape[1]}` columns"))

    # First 5 rows
    display(Markdown("**First 5 rows:**"))
    display(df.head())

    # Last 5 rows
    display(Markdown("**Last 5 rows:**"))
    display(df.tail())

    # Column names and types
    col_info = pd.DataFrame({
        "Column": df.columns,
        "Dtype": df.dtypes.values,
        "Non-Null Count": df.notna().sum().values,
        "Null Count": df.isna().sum().values,
    })
    display(Markdown("**Column Names & Data Types:**"))
    display(col_info)

    # Memory usage
    mem_bytes = df.memory_usage(deep=True).sum()
    display(Markdown(
        f"**Memory Usage:** `{mem_bytes / 1024:.1f}` KB "
        f"(`{mem_bytes / (1024**2):.3f}` MB)"
    ))

    # Summary statistics
    numeric_df = df.select_dtypes(include=[np.number])
    if not numeric_df.empty:
        display(Markdown("**Summary Statistics (numeric columns):**"))
        display(numeric_df.describe())
    else:
        display(Markdown("*No numeric columns to summarize.*"))

    print("\n" + "─" * 70 + "\n")

In [16]:
# Display overview for each parsed table across all experiments
for exp_label, tables in parsed_data.items():
    display(Markdown(f"## 📂 {exp_label}"))
    display(Markdown("---"))
    for tname, tdata in tables.items():
        if isinstance(tdata, pd.DataFrame) and not tdata.empty:
            dataset_overview(tdata, f"{exp_label} — {tname}")

## 📂 Experiment 1

---

### Experiment 1 — seedlings

**Shape:** `10` rows × `12` columns

**First 5 rows:**

...

**Column Names & Data Types:**

,Column,Dtype,Non-Null Count,Null Count
0,experiment,str,83,0
1,system,str,83,0
2,Plant No.,object,77,6
3,Plant height (cm),object,83,0
4,Shoot length (cm),object,83,0
5,Root length (cm),object,83,0
6,H.D (cm),str,77,6
7,Total weight (g),object,83,0
8,Stem diameter (cm),object,83,0
9,Shoot weight before removing wilted ...,object,83,0


**Memory Usage:** `44.2` KB (`0.043` MB)

*No numeric columns to summarize.*


──────────────────────────────────────────────────────────────────────



---

## 5 · Data Quality Assessment

We perform a thorough quality audit across **every** parsed table. This section checks for:

| Check | Purpose |
|---|---|
| Missing values | Identify nulls and their distribution |
| Duplicate rows | Detect exact row duplicates |
| Constant columns | Find zero-variance columns |
| Unique value counts | Understand cardinality |
| Invalid datatypes | Detect string values in expected numeric columns |
| Negative values | Flag suspicious negatives in measurement columns |
| Impossible values | Flag physically impossible measurements |
| Outliers | Detect statistical outliers (IQR method) — **flag only, never remove** |
| Null percentages | Rank columns by missing data severity |

In [17]:
# ==========================================================================
# Quality assessment utility functions
# ==========================================================================

def check_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame summarizing missing values per column."""
    missing = df.isnull().sum()
    pct = (missing / len(df) * 100).round(2)
    result = pd.DataFrame({
        "missing_count": missing,
        "missing_pct": pct,
        "dtype": df.dtypes,
    })
    return result[result["missing_count"] > 0].sort_values("missing_pct", ascending=False)


def check_duplicates(df: pd.DataFrame) -> int:
    """Return count of exact duplicate rows."""
    return int(df.duplicated().sum())


def check_constant_columns(df: pd.DataFrame) -> List[str]:
    """Identify columns with only one unique value (excluding NaN)."""
    constant_cols = []
    for col in df.columns:
        n_unique = df[col].dropna().nunique()
        if n_unique <= 1:
            constant_cols.append(col)
    return constant_cols


def check_unique_values(df: pd.DataFrame) -> pd.DataFrame:
    """Return a summary of unique value counts per column."""
    return pd.DataFrame({
        "unique_count": df.nunique(),
        "total_rows": len(df),
        "unique_pct": (df.nunique() / len(df) * 100).round(2),
        "sample_values": [
            str(df[col].dropna().unique()[:5].tolist()) for col in df.columns
        ],
    })


def check_invalid_numeric(
    df: pd.DataFrame, expected_numeric_cols: Optional[List[str]] = None
) -> Dict[str, List]:
    """
    Detect non-numeric values in columns that should be numeric.
    If expected_numeric_cols is None, checks all object-type columns
    for values that look partially numeric.
    """
    issues = {}
    cols_to_check = expected_numeric_cols or [
        c for c in df.columns if df[c].dtype == object
    ]

    for col in cols_to_check:
        if col not in df.columns:
            continue
        non_null = df[col].dropna()
        if non_null.empty:
            continue
        converted = pd.to_numeric(non_null, errors="coerce")
        failed_mask = converted.isna() & non_null.notna()
        if failed_mask.any():
            bad_vals = non_null[failed_mask].unique().tolist()
            if len(bad_vals) > 0:
                issues[col] = bad_vals[:10]

    return issues


def check_negative_values(df: pd.DataFrame) -> Dict[str, int]:
    """Flag columns with negative values (suspicious for physical measurements)."""
    negatives = {}
    for col in df.select_dtypes(include=[np.number]).columns:
        neg_count = int((df[col] < 0).sum())
        if neg_count > 0:
            negatives[col] = neg_count
    return negatives


def check_outliers_iqr(df: pd.DataFrame, factor: float = 1.5) -> pd.DataFrame:
    """
    Detect outliers using the IQR method.
    Returns a summary of outlier counts per numeric column.
    NOTE: Outliers are flagged only — never removed.
    """
    records = []
    for col in df.select_dtypes(include=[np.number]).columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - factor * iqr
        upper = q3 + factor * iqr
        outlier_mask = (df[col] < lower) | (df[col] > upper)
        n_outliers = int(outlier_mask.sum())
        if n_outliers > 0:
            records.append({
                "column": col,
                "q1": round(q1, 4),
                "q3": round(q3, 4),
                "iqr": round(iqr, 4),
                "lower_bound": round(lower, 4),
                "upper_bound": round(upper, 4),
                "n_outliers": n_outliers,
                "pct_outliers": round(n_outliers / len(df) * 100, 2),
            })
    return pd.DataFrame(records) if records else pd.DataFrame()


logger.info("Quality assessment functions defined.")

2026-07-15 19:48:59,093 | INFO | Quality assessment functions defined.


In [18]:
def run_quality_assessment(
    df: pd.DataFrame, title: str, show_details: bool = True
) -> Dict[str, Any]:
    """
    Run the complete quality assessment suite on a DataFrame.
    Returns a dict of all findings for downstream use.
    """
    results = {}

    if show_details:
        display(Markdown(f"### 🔍 {title}"))

    # 1. Missing values
    missing_df = check_missing_values(df)
    results["missing"] = missing_df
    if show_details:
        if not missing_df.empty:
            display(Markdown("**Missing Values:**"))
            display(missing_df)
        else:
            display(Markdown("**Missing Values:** ✅ None detected."))

    # 2. Duplicates
    dup_count = check_duplicates(df)
    results["duplicates"] = dup_count
    if show_details:
        if dup_count > 0:
            display(Markdown(f"**Duplicate Rows:** ⚠️ **{dup_count}** exact duplicates found."))
        else:
            display(Markdown("**Duplicate Rows:** ✅ None detected."))

    # 3. Constant columns
    const_cols = check_constant_columns(df)
    results["constant_columns"] = const_cols
    if show_details:
        if const_cols:
            display(Markdown(
                f"**Constant Columns:** ⚠️ {len(const_cols)} column(s) with ≤1 unique value: "
                f"`{'`, `'.join(map(str, const_cols))}`"
            ))
        else:
            display(Markdown("**Constant Columns:** ✅ None detected."))

    # 4. Unique values
    unique_df = check_unique_values(df)
    results["unique_values"] = unique_df
    if show_details:
        display(Markdown("**Unique Value Counts:**"))
        display(unique_df)

    # 5. Invalid numeric values
    invalid_numeric = check_invalid_numeric(df)
    results["invalid_numeric"] = invalid_numeric
    if show_details:
        if invalid_numeric:
            display(Markdown("**Invalid Numeric Values (non-numeric in expected numeric columns):**"))
            for col, vals in invalid_numeric.items():
                display(Markdown(f"- `{col}`: `{vals}` ({len(vals)} unique invalid value(s))"))
        else:
            display(Markdown("**Invalid Numeric Values:** ✅ None detected."))

    # 6. Negative values
    negatives = check_negative_values(df)
    results["negatives"] = negatives
    if show_details:
        if negatives:
            display(Markdown("**Negative Values (suspicious for physical measurements):**"))
            for col, count in negatives.items():
                display(Markdown(f"- `{col}`: {count} negative value(s)"))
        else:
            display(Markdown("**Negative Values:** ✅ None detected in numeric columns."))

    # 7. Outliers (IQR)
    outlier_df = check_outliers_iqr(df)
    results["outliers"] = outlier_df
    if show_details:
        if not outlier_df.empty:
            display(Markdown(
                "**Outliers (IQR Method, factor=1.5):**  \\n"
                "> ⚠️ Flagged for review only — **not removed**."
            ))
            display(outlier_df)
        else:
            display(Markdown("**Outliers (IQR):** ✅ None detected."))

    if show_details:
        print("\n" + "─" * 70 + "\n")

    return results

In [19]:
# ==========================================================================
# Run quality assessment on ALL parsed tables
# ==========================================================================
quality_reports: Dict[str, Dict[str, Dict]] = {}

for exp_label, tables in parsed_data.items():
    display(Markdown(f"## 📂 {exp_label} — Quality Assessment"))
    display(Markdown("---"))
    quality_reports[exp_label] = {}

    for tname, tdata in tables.items():
        if isinstance(tdata, pd.DataFrame) and not tdata.empty:
            quality_reports[exp_label][tname] = run_quality_assessment(
                tdata, f"{exp_label} — {tname}"
            )

logger.info("Quality assessment complete for all datasets.")

## 📂 Experiment 1 — Quality Assessment

---

### 🔍 Experiment 1 — seedlings

**Missing Values:**

,missing_count,missing_pct,dtype
Date,9,90.0000,object


...

- `No.of leaves`: `['No.of leaves ']` (1 unique invalid value(s))

**Negative Values:** ✅ None detected in numeric columns.

**Outliers (IQR):** ✅ None detected.

2026-07-15 19:49:09,850 | INFO | Quality assessment complete for all datasets.



──────────────────────────────────────────────────────────────────────



---

## 6 · Cleaning

### Cleaning Philosophy

We apply **only safe, reversible, non-destructive** transformations:

| ✅ Allowed | ❌ Not Allowed |
|---|---|
| Trim whitespace from strings | Delete rows due to outliers |
| Remove exact duplicate rows | Remove suspicious values |
| Standardize column names (snake_case) | Fill missing values without justification |
| Fix obvious datatype issues | Drop columns with partial data |
| Convert date strings to datetime | Impute scientific measurements |
| Replace dash/hyphen placeholders with NaN | Modify measurement values |

Every cleaning step shows: **reason**, **before**, and **after**.

In [20]:
# ==========================================================================
# Cleaning utility functions
# ==========================================================================

def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize column names to lowercase snake_case.
    Removes trailing/leading whitespace, replaces spaces and special chars.
    """
    df = df.copy()
    new_cols = []
    for col in df.columns:
        c = str(col).strip()
        c = c.lower()
        c = c.replace(" ", "_").replace(".", "").replace("(", "").replace(")", "")
        c = c.replace("__", "_").replace("___", "_")
        c = c.rstrip("_")
        new_cols.append(c)
    df.columns = new_cols
    return df


def trim_string_whitespace(df: pd.DataFrame) -> pd.DataFrame:
    """
    Trim leading/trailing whitespace from all string columns.
    """
    df = df.copy()
    str_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip()
        # Restore NaN for values that were 'nan' strings
        df[col] = df[col].replace({"nan": np.nan, "NaT": np.nan, "": np.nan, "None": np.nan})
    return df


def replace_dash_placeholders(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replace dash/hyphen placeholder values with NaN.
    These are commonly used in the raw data to indicate no measurement.
    """
    df = df.copy()
    dash_patterns = ["-", "--", "- ", " -", "N/A", "n/a", "NA", "na"]
    df = df.replace(dash_patterns, np.nan)
    return df


def remove_exact_duplicates(df: pd.DataFrame) -> Tuple[pd.DataFrame, int]:
    """
    Remove exact duplicate rows. Returns (cleaned_df, n_removed).
    """
    n_before = len(df)
    df_clean = df.drop_duplicates().reset_index(drop=True)
    n_removed = n_before - len(df_clean)
    return df_clean, n_removed


def remove_fully_empty_rows(df: pd.DataFrame) -> Tuple[pd.DataFrame, int]:
    """
    Remove rows where ALL values (excluding the 'experiment' label column) are NaN.
    Returns (cleaned_df, n_removed).
    """
    non_label_cols = [c for c in df.columns if c != "experiment"]
    mask = df[non_label_cols].notna().any(axis=1)
    n_removed = int((~mask).sum())
    return df[mask].reset_index(drop=True), n_removed


def try_convert_dates(df: pd.DataFrame, date_cols: Optional[List[str]] = None) -> pd.DataFrame:
    """
    Attempt to convert date columns to datetime.
    If date_cols is None, auto-detects columns with 'date' in the name.
    """
    df = df.copy()
    if date_cols is None:
        date_cols = [c for c in df.columns if "date" in c.lower()]

    for col in date_cols:
        if col not in df.columns:
            continue
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            continue
        try:
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
            logger.info(f"  Converted '{col}' to datetime.")
        except Exception as e:
            logger.warning(f"  Could not convert '{col}' to datetime: {e}")
    return df


def try_convert_numeric(df: pd.DataFrame, exclude_cols: Optional[List[str]] = None) -> pd.DataFrame:
    """
    Attempt to convert object columns to numeric where possible.
    Columns where conversion fails for >50% of non-null values are left as-is.
    """
    df = df.copy()
    exclude = set(exclude_cols or [])
    exclude.update({"experiment", "replicate", "system", "day"})

    for col in df.select_dtypes(include=["object"]).columns:
        if col in exclude:
            continue
        non_null = df[col].dropna()
        if non_null.empty:
            continue
        converted = pd.to_numeric(non_null, errors="coerce")
        success_rate = converted.notna().sum() / len(non_null)
        if success_rate > 0.5:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            logger.info(
                f"  Converted '{col}' to numeric "
                f"({success_rate:.0%} success, {(1-success_rate):.0%} coerced to NaN)."
            )
    return df


logger.info("Cleaning utility functions defined.")

2026-07-15 19:50:02,469 | INFO | Cleaning utility functions defined.


In [21]:
def clean_dataframe(
    df: pd.DataFrame, table_name: str, exp_label: str
) -> pd.DataFrame:
    """
    Apply the full cleaning pipeline to a single DataFrame.
    Documents every step with before/after comparisons.
    """
    display(Markdown(f"### 🧹 Cleaning: {exp_label} — {table_name}"))

    original_shape = df.shape
    df_clean = df.copy()

    # ── Step 1: Standardize column names ──
    display(Markdown("**Step 1: Standardize column names to snake_case**"))
    display(Markdown("*Reason:* Consistent naming convention for downstream processing."))
    cols_before = list(df_clean.columns)
    df_clean = standardize_column_names(df_clean)
    cols_after = list(df_clean.columns)

    changed = [(b, a) for b, a in zip(cols_before, cols_after) if b != a]
    if changed:
        rename_df = pd.DataFrame(changed, columns=["Before (Raw)", "After (Snake Case)"])
        display(rename_df.head(15))
        if len(changed) > 15:
            print(f"  ... and {len(changed) - 15} more.")
    else:
        display(Markdown("*Before:* " + ", ".join([f"`{c}`" for c in cols_before[:5]]) + ("..." if len(cols_before) > 5 else "")))
        display(Markdown("*After:* " + ", ".join([f"`{c}`" for c in cols_after[:5]]) + ("..." if len(cols_after) > 5 else "")))
        print("  No changes needed.")

    # ── Step 2: Trim whitespace ──
    display(Markdown("**Step 2: Trim string whitespace**"))
    display(Markdown("*Reason:* Remove invisible whitespace that causes false mismatches."))
    
    str_cols = df_clean.select_dtypes(include=["object", "string"]).columns
    trimmed_examples = []
    for col in str_cols:
        non_null_mask = df_clean[col].notna()
        if not non_null_mask.any():
            continue
        vals_before = df_clean.loc[non_null_mask, col].astype(str)
        vals_after = vals_before.str.strip()
        diff_mask = vals_before != vals_after
        if diff_mask.any():
            for val_b, val_a in zip(vals_before[diff_mask].unique(), vals_after[diff_mask].unique()):
                trimmed_examples.append((col, repr(val_b), repr(val_a)))
                
    df_clean = trim_string_whitespace(df_clean)
    
    if trimmed_examples:
        trim_df = pd.DataFrame(trimmed_examples[:15], columns=["Column", "Before (Untrimmed)", "After (Trimmed)"])
        display(trim_df)
        if len(trimmed_examples) > 15:
            print(f"  ... and {len(trimmed_examples) - 15} more whitespace modifications.")
    else:
        if len(str_cols) > 0:
            sample_col = str_cols[0]
            sample_val = df_clean[sample_col].dropna().iloc[0] if df_clean[sample_col].dropna().shape[0] > 0 else "N/A"
            display(Markdown(f"*Before:* `{repr(sample_val)}` (in `{sample_col}`)"))
            display(Markdown(f"*After:* `{repr(sample_val)}` (in `{sample_col}`)"))
        print("  No leading/trailing whitespace detected.")

    # ── Step 3: Replace dash placeholders ──
    display(Markdown("**Step 3: Replace dash/placeholder values with NaN**"))
    display(Markdown("*Reason:* Dashes ('-') and placeholders are used as 'no data' markers, not valid measurements."))
    
    dash_patterns = ["-", "--", "- ", " -", "N/A", "n/a", "NA", "na"]
    placeholder_changes = []
    for col in df_clean.columns:
        mask = df_clean[col].isin(dash_patterns)
        if mask.any():
            found_vals = df_clean.loc[mask, col].unique()
            count = mask.sum()
            for val in found_vals:
                placeholder_changes.append((col, f"'{val}' (count: {count})", "NaN"))
                
    null_before = int(df_clean.isnull().sum().sum())
    df_clean = replace_dash_placeholders(df_clean)
    null_after = int(df_clean.isnull().sum().sum())
    new_nulls = null_after - null_before
    
    if placeholder_changes:
        placeholder_df = pd.DataFrame(placeholder_changes, columns=["Column", "Before (Placeholder)", "After (Converted)"])
        display(placeholder_df)
    else:
        display(Markdown(f"*Before:* {null_before:,} NaN values"))
        display(Markdown(f"*After:* {null_after:,} NaN values"))
        print("  No dash/placeholder values detected.")

    # ── Step 4: Remove fully empty rows ──
    display(Markdown("**Step 4: Remove fully empty rows**"))
    display(Markdown("*Reason:* Rows with no data (all NaN) are structural artifacts from Excel."))
    rows_before = len(df_clean)
    df_clean, n_empty = remove_fully_empty_rows(df_clean)
    
    display(Markdown(f"| Metric | Before | After | Removed |"))
    display(Markdown(f"|---|---|---|---|"))
    display(Markdown(f"| Row Count | {rows_before:,} | {len(df_clean):,} | {n_empty:,} |"))

    # ── Step 5: Remove exact duplicates ──
    display(Markdown("**Step 5: Remove exact duplicate rows**"))
    display(Markdown("*Reason:* Exact duplicates are data entry artifacts, not real observations."))
    rows_before = len(df_clean)
    df_clean, n_dups = remove_exact_duplicates(df_clean)
    
    display(Markdown(f"| Metric | Before | After | Removed |"))
    display(Markdown(f"|---|---|---|---|"))
    display(Markdown(f"| Row Count | {rows_before:,} | {len(df_clean):,} | {n_dups:,} |"))

    # ── Step 6: Convert dates ──
    display(Markdown("**Step 6: Convert date columns to datetime**"))
    display(Markdown("*Reason:* Enable time-series analysis and proper sorting."))
    
    date_cols = [c for c in df_clean.columns if "date" in c.lower()]
    dtypes_before = {col: str(df_clean[col].dtype) for col in date_cols if col in df_clean.columns}
    
    df_clean = try_convert_dates(df_clean)
    
    if date_cols:
        date_changes = []
        for col in date_cols:
            if col in df_clean.columns:
                date_changes.append((col, dtypes_before.get(col, "unknown"), str(df_clean[col].dtype)))
        
        date_df = pd.DataFrame(date_changes, columns=["Column", "Before (Dtype)", "After (Dtype)"])
        display(date_df)
    else:
        display(Markdown("*Before:* No date columns"))
        display(Markdown("*After:* No date columns"))
        print("  No date columns detected.")

    # ── Step 7: Convert numeric columns ──
    display(Markdown("**Step 7: Convert object columns to numeric where appropriate**"))
    display(Markdown(
        "*Reason:* Some numeric measurements are stored as strings due to Excel parsing. "
        "Columns with >50% numeric success rate are converted; non-convertible values become NaN."
    ))
    text_keep = [c for c in df_clean.columns if "diameter" in c.lower() and "head" in c.lower()]
    
    obj_cols = [c for c in df_clean.select_dtypes(include=["object"]).columns if c not in text_keep and c not in {"experiment", "replicate", "system", "day"}]
    dtypes_before = {col: str(df_clean[col].dtype) for col in obj_cols}
    
    df_clean = try_convert_numeric(df_clean, exclude_cols=text_keep)
    
    numeric_changes = []
    for col in obj_cols:
        if col in df_clean.columns:
            dtype_after = str(df_clean[col].dtype)
            numeric_changes.append((col, dtypes_before[col], dtype_after))
                
    if numeric_changes:
        num_df = pd.DataFrame(numeric_changes, columns=["Column", "Before (Dtype)", "After (Dtype)"])
        display(num_df)
    else:
        display(Markdown("*Before:* No object columns to convert"))
        display(Markdown("*After:* No object columns to convert"))
        print("  No object columns converted to numeric.")

    # ── Summary ──
    display(Markdown(f"**Cleaning Summary for {table_name}:**"))
    print(f"  Original shape: {original_shape}")
    print(f"  Cleaned shape:  {df_clean.shape}")
    print(f"  Rows removed:   {original_shape[0] - df_clean.shape[0]}")
    print("\n" + "─" * 70 + "\n")

    return df_clean

In [22]:
# ==========================================================================
# Apply cleaning to ALL parsed tables
# ==========================================================================
cleaned_data: Dict[str, Dict[str, pd.DataFrame]] = {}

for exp_label, tables in parsed_data.items():
    display(Markdown(f"## 📂 {exp_label} — Cleaning"))
    display(Markdown("---"))
    cleaned_data[exp_label] = {}

    for tname, tdata in tables.items():
        if isinstance(tdata, pd.DataFrame) and not tdata.empty:
            cleaned_data[exp_label][tname] = clean_dataframe(
                tdata, tname, exp_label
            )

logger.info("Cleaning complete for all datasets.")
print("\n✅ All datasets cleaned.")

## 📂 Experiment 1 — Cleaning

---

### 🧹 Cleaning: Experiment 1 — seedlings

**Step 1: Standardize column names to snake_case**

*Reason:* Consistent naming convention for downstream processing.

...

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18536\4227539317.py:140: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = [c for c in df_clean.select_dtypes(include=["object"]).columns if c not in text_keep and c not in {"experiment", "replicate", "system", "day"}]
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18536\1126182943.py:99: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them

,Column,Before (Dtype),After (Dtype)
0,plant_no,str,float64
1,plant_height_cm,str,float64
2,shoot_length_cm,str,float64
3,root_length_cm,str,float64
4,hd_cm,str,str
5,total_weight_g,str,float64
6,stem_diameter_cm,str,float64
7,shoot_weight_before_removing_wilted_...,str,float64
8,shoot_weight_after_removing_wilted_l...,str,float64
9,root_weight_g,str,float64


**Cleaning Summary for harvest:**

2026-07-15 19:50:19,589 | INFO | Cleaning complete for all datasets.


  Original shape: (83, 13)
  Cleaned shape:  (79, 13)
  Rows removed:   4

──────────────────────────────────────────────────────────────────────


✅ All datasets cleaned.


---

## 7 · Validation

After cleaning, we re-run the **entire quality assessment suite** on the cleaned data to:
1. Confirm that cleaning steps had the desired effect.
2. Verify no new issues were introduced.
3. Document the remaining data quality state.

> Any remaining issues flagged here are documented but **not automatically fixed** — they require domain-expert review.

In [23]:
# ==========================================================================
# Post-cleaning validation
# ==========================================================================
validation_reports: Dict[str, Dict[str, Dict]] = {}

for exp_label, tables in cleaned_data.items():
    display(Markdown(f"## ✅ {exp_label} — Post-Cleaning Validation"))
    display(Markdown("---"))
    validation_reports[exp_label] = {}

    for tname, tdata in tables.items():
        if isinstance(tdata, pd.DataFrame) and not tdata.empty:
            validation_reports[exp_label][tname] = run_quality_assessment(
                tdata, f"{exp_label} — {tname} (POST-CLEANING)"
            )

logger.info("Post-cleaning validation complete.")

## ✅ Experiment 1 — Post-Cleaning Validation

---

### 🔍 Experiment 1 — seedlings (POST-CLEANING)

**Missing Values:**

,missing_count,missing_pct,dtype
date,9,90.0000,datetime64[us]


...

**Negative Values:** ✅ None detected in numeric columns.

**Outliers (IQR Method, factor=1.5):**  \n> ⚠️ Flagged for review only — **not removed**.

,column,q1,q3,iqr,lower_bound,upper_bound,n_outliers,pct_outliers
0,plant_height_cm,60.0000,72.8125,12.8125,40.7812,92.0312,1,1.2700
1,shoot_length_cm,17.0000,19.0000,2.0000,14.0000,22.0000,1,1.2700
2,root_length_cm,42.0000,54.8125,12.8125,22.7812,74.0312,1,1.2700
3,stem_diameter_cm,1.0000,1.0000,0.0000,1.0000,1.0000,17,21.5200
4,root_weight_g,29.7500,39.0000,9.2500,15.8750,52.8750,5,6.3300


2026-07-15 19:51:08,089 | INFO | Post-cleaning validation complete.



──────────────────────────────────────────────────────────────────────



In [24]:
# ==========================================================================
# Compare pre- vs. post-cleaning metrics
# ==========================================================================

display(Markdown("### 📊 Before vs. After Cleaning Comparison"))

comparison_records = []
for exp_label in quality_reports:
    for tname in quality_reports[exp_label]:
        pre = quality_reports[exp_label].get(tname, {})
        post = validation_reports.get(exp_label, {}).get(tname, {})

        if not pre or not post:
            continue

        orig_table = parsed_data[exp_label].get(tname)
        clean_table = cleaned_data[exp_label].get(tname)

        orig_rows = orig_table.shape[0] if isinstance(orig_table, pd.DataFrame) else 0
        clean_rows = clean_table.shape[0] if isinstance(clean_table, pd.DataFrame) else 0

        pre_missing_df = pre.get("missing", pd.DataFrame())
        pre_missing = int(pre_missing_df["missing_count"].sum()) if not pre_missing_df.empty else 0
        post_missing_df = post.get("missing", pd.DataFrame())
        post_missing = int(post_missing_df["missing_count"].sum()) if not post_missing_df.empty else 0

        comparison_records.append({
            "Experiment": exp_label,
            "Table": tname,
            "Rows (Before)": orig_rows,
            "Rows (After)": clean_rows,
            "Rows Removed": orig_rows - clean_rows,
            "Duplicates (Before)": pre.get("duplicates", 0),
            "Duplicates (After)": post.get("duplicates", 0),
            "Total NaN (Before)": pre_missing,
            "Total NaN (After)": post_missing,
        })

if comparison_records:
    comparison_df = pd.DataFrame(comparison_records)
    display(comparison_df)
else:
    print("No comparison data available.")

### 📊 Before vs. After Cleaning Comparison

,Experiment,Table,Rows (Before),Rows (After),Rows Removed,Duplicates (Before),Duplicates (After),Total NaN (Before),Total NaN (After)
0,Experiment 1,seedlings,10,10,0,0,0,9,9
1,Experiment 1,sensor_water_quality,641,641,0,0,0,3850,4299
2,Experiment 1,portable_water_quality,33,32,1,0,0,607,788
3,Experiment 1,nutrients_date,24,5,19,18,2,19,3
4,Experiment 1,nutrients_nutrient_solution_addition...,24,5,19,18,0,114,0
5,Experiment 1,nutrients_acid_consumption_(ml),24,4,20,19,0,120,0
6,Experiment 1,nutrients_water_consumption_l,24,5,19,18,0,114,0
7,Experiment 1,head_diameter,300,204,96,96,0,240,48
8,Experiment 1,harvest,73,73,0,0,0,10,10
9,Experiment 2,seedlings,12,12,0,0,0,12,12


---

## Section 9 - Data Dictionary

Automatically generated professional data dictionaries for every cleaned dataset from each experiment. Each dictionary includes metadata about columns, data types, null counts/percentages, unique value counts, sample values, and descriptions.

In [25]:
KNOWN_DESCRIPTIONS = {
    "experiment": "Experiment identifier (Experiment 1, 2, or 3)",
    "replicate": "Replicate/treatment system identifier",
    "system": "Replicate-Treatment system code (e.g., R1T1)",
    "date": "Date of measurement",
    "date_": "Date of measurement",
    "plant_no": "Plant identification number within replicate",
    "plant_no_": "Plant identification number within replicate",
    "plant_height__cm": "Total plant height in centimeters",
    "plant_height_cm": "Total plant height in centimeters",
    "shoot_length_cm": "Shoot (above-root) length in centimeters",
    "root_length_cm": "Root length in centimeters",
    "head_diameter_cm": "Head diameter in centimeters (often W*H format)",
    "hd_cm": "Head diameter in centimeters (often W*H format)",
    "stem_diameter__cm": "Stem diameter in centimeters",
    "stem_diameter_cm": "Stem diameter in centimeters",
    "total_weight_g": "Total plant weight in grams",
    "shoot_weight_g": "Shoot weight in grams",
    "root_weight_g": "Root weight in grams",
    "root_weight_g_": "Root weight in grams",
    "no_of_leaves": "Number of leaves counted",
    "no_of_leaves_": "Number of leaves counted",
    "noof_leaves_": "Number of leaves counted",
    "ph": "pH level of nutrient solution",
    "ph_": "pH level of nutrient solution",
    "ec": "Electrical conductivity (mS/cm)",
    "ec_": "Electrical conductivity (mS/cm)",
    "tds": "Total dissolved solids (ppm)",
    "tds_": "Total dissolved solids (ppm)",
    "water_temp": "Water temperature (°C)",
    "water_temp_": "Water temperature (°C)",
    "air_temp": "Air temperature (°C)",
    "air_temp_": "Air temperature (°C)",
    "air_temp_c": "Air temperature (°C)",
    "rh%": "Relative humidity percentage",
    "rh_": "Relative humidity percentage",
    "rh_%": "Relative humidity percentage",
    "co2": "CO₂ concentration",
    "co2_": "CO₂ concentration",
    "co2_ppm": "CO₂ concentration in parts per million",
    "tme": "Time of measurement",
    "tme_": "Time of measurement",
    "day": "Day of measurement relative to transplant",
    "head_diameter": "Head diameter (often W*H format)",
    "shoot_weight_before_removing_wilted_leaves_g": "Shoot weight before removing wilted leaves (g)",
    "shoot_weight_after_removing_wilted_leavesg": "Shoot weight after removing wilted leaves (g)",
    "shoot_weight_before_removing_wilted_leaves_g_": "Shoot weight before removing wilted leaves (g)",
    "shoot_weight_after_removing_wilted_leaves_g_": "Shoot weight after removing wilted leaves (g)",
}

def generate_data_dictionary(df: pd.DataFrame) -> pd.DataFrame:
    """Generate a professional data dictionary for a DataFrame."""
    records = []
    for col in df.columns:
        col_lower = col.lower().replace(" ", "_")
        description = KNOWN_DESCRIPTIONS.get(col_lower, KNOWN_DESCRIPTIONS.get(col, ""))
        
        records.append({
            "Column Name": col,
            "Data Type": str(df[col].dtype),
            "Missing Count": int(df[col].isna().sum()),
            "Missing Percentage": round(df[col].isna().mean() * 100, 2),
            "Unique Values": int(df[col].nunique()),
            "Sample Values": str(df[col].dropna().unique()[:5].tolist()),
            "Description": description if description else ""
        })
    return pd.DataFrame(records)

# Create reports/data_dictionary/ directory
dict_dir = PROJECT_ROOT / "reports" / "data_dictionary"
dict_dir.mkdir(parents=True, exist_ok=True)

# Generate, display, and save data dictionaries
for exp_label, tables in cleaned_data.items():
    exp_idx = exp_label.split()[-1]
    display(Markdown(f"### 📂 {exp_label} — Data Dictionaries"))
    
    for tname, tdata in tables.items():
        if isinstance(tdata, pd.DataFrame) and not tdata.empty:
            dd_df = generate_data_dictionary(tdata)
            
            # Display
            display(Markdown(f"#### {exp_label} — {tname}"))
            display(dd_df)
            print()
            
            # Save
            csv_path = dict_dir / f"exp{exp_idx}_{tname}_dict.csv"
            dd_df.to_csv(csv_path, index=False)
            logger.info(f"Saved data dictionary to: {csv_path}")

print("\n✅ All data dictionaries generated and saved.")

### 📂 Experiment 1 — Data Dictionaries

#### Experiment 1 — seedlings

,Column Name,Data Type,Missing Count,Missing Percentage,Unique Values,Sample Values,Description
0,experiment,str,0,0.0000,1,['Experiment 1'],"Experiment identifier (Experiment 1,..."
1,date,datetime64[us],9,90.0000,1,[Timestamp('2024-03-09 00:00:00')],Date of measurement
2,plant_no,int64,0,0.0000,10,"[1, 2, 3, 4, 5]",Plant identification number within r...
3,plant_height_cm,float64,0,0.0000,4,"[9.0, 10.0, 9.5, 12.0]",Total plant height in centimeters
4,shoot_length_cm,float64,0,0.0000,3,"[2.0, 3.0, 2.5]",Shoot (above-root) length in centime...
5,root_length_cm,float64,0,0.0000,3,"[7.0, 6.5, 9.0]",Root length in centimeters
6,head_diameter_cm,str,0,0.0000,7,"['4*2', '3.5*3', '3.5*2.5', '3*3', '...",Head diameter in centimeters (often ...
7,stem_diameter_cm,float64,0,0.0000,1,[0.1],Stem diameter in centimeters
8,total_weight_g,float64,0,0.0000,7,"[1.3, 1.2, 1.1, 1.14, 1.0]",Total plant weight in grams
9,shoot_weight_g,float64,0,0.0000,9,"[0.5, 0.55, 0.53, 0.45, 0.4]",Shoot weight in grams


2026-07-15 19:51:58,570 | INFO | Saved data dictionary to: e:\HydroGrow-AI\reports\data_dictionary\exp1_seedlings_dict.csv


#### Experiment 1 — sensor_water_quality

...

2026-07-15 19:51:59,029 | INFO | Saved data dictionary to: e:\HydroGrow-AI\reports\data_dictionary\exp3_nutrients_acid_consumption_ml_dict.csv


#### Experiment 3 — harvest

,Column Name,Data Type,Missing Count,Missing Percentage,Unique Values,Sample Values,Description
0,experiment,str,0,0.0000,1,['Experiment 3'],"Experiment identifier (Experiment 1,..."
1,system,str,0,0.0000,7,"['R1T1', 'System', 'R2T1', 'R3T1', '...",Replicate-Treatment system code (e.g...
2,plant_no,float64,7,8.8600,12,"[1.0, 2.0, 3.0, 4.0, 5.0]",Plant identification number within r...
3,plant_height_cm,float64,1,1.2700,33,"[60.0, 75.0, 100.0, 72.0, 76.0]",Total plant height in centimeters
4,shoot_length_cm,float64,1,1.2700,14,"[20.0, 19.0, 17.0, 15.0, 16.0]",Shoot (above-root) length in centime...
5,root_length_cm,float64,1,1.2700,35,"[40.0, 41.0, 58.0, 83.0, 57.0]",Root length in centimeters
6,hd_cm,str,6,7.5900,51,"['30*31', '29*31', '28*30', '29*29',...",Head diameter in centimeters (often ...
7,total_weight_g,float64,1,1.2700,61,"[328.0, 252.0, 294.0, 331.0, 357.0]",Total plant weight in grams
8,stem_diameter_cm,float64,1,1.2700,9,"[1.0, 1.5, 1.2, 1.141666667, 1.04166...",Stem diameter in centimeters
9,shoot_weight_before_removing_wilted_...,float64,1,1.2700,64,"[271.0, 244.0, 239.0, 295.0, 317.0]",Shoot weight before removing wilted ...


2026-07-15 19:51:59,051 | INFO | Saved data dictionary to: e:\HydroGrow-AI\reports\data_dictionary\exp3_harvest_dict.csv




✅ All data dictionaries generated and saved.
